# A transformer block, one piece at a time

Runs the two sublayers of a decoder block — self-attention, then position-wise
feed-forward — with a toy config so every printout stays readable. The wiring
built up here by hand is exactly `TransformerBlock` in
`models/blocks_of_transformer.py` (attention + residuals + LayerNorm + dropout).

**Roadmap**
1. Setup — seed + import
2. Attention & FeedForward on toy data
3. Layer normalization
4. Pre-LN vs Post-LN — the GPT-2 choice

(Residual connections are explained in the markdown between sections 2 and 3.)


In [ ]:
# torch.manual_seed(0): identical tensors on every rerun — a notebook that
# explains specific shapes and values must be reproducible.
# The ROOT walk: jupyter's cwd depends on how the session was launched, so we
# search upward for haytham_llm/config.py and prepend the repo root to sys.path.
import math
import pathlib
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F

def p(*text):
    print(*text)

ROOT = pathlib.Path.cwd()

while not (ROOT / "haytham_llm" / "config.py").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
    
sys.path.insert(0,str(ROOT))

torch.manual_seed(0)
print(ROOT)

## 2. Attention & FeedForward on toy data

In [ ]:
# Toy hyper-parameters vs the real defaults in config.py —
# d_model 6 (vs 768), n_heads 2 (vs 12) -> per-head width d_k = 3 (vs 64),
# vocab 32, seq_len 16. Small enough to print every number, big enough to
# exercise all the reshape/transpose logic downstream.
# x stands in for embedding output: (batch, seq_len, d_model).
from haytham_llm.models.attention import MultiHeadAttention
from haytham_llm.models.feedforward import FeedForward
from haytham_llm.config import ModelConfig

TOKENS = ["the","cat",'sat','on','mat']

config =  ModelConfig(
 vocab_size=32,
 seq_len=16,
 d_model= 6,
 n_heads= 2,
 n_layers=2,
 qkv_bias=False,
 dropout =0.1
)

batch = 1
seq_len = len(TOKENS)

x = torch.rand(batch,seq_len,config.d_model)

print (config)

print(x.shape,x)



In [ ]:
# Self-attention lets tokens exchange information across the sequence; the
# FeedForward then runs a per-token MLP (no cross-token mixing).
# Both are shape-preserving (B, S, D) -> (B, S, D): they transform without
# changing x's shape — which is exactly what lets residuals add them later.
# attention() also returns an updated kv_cache; that only matters for
# generation (see attention_is_all_you_need.ipynb) and is unused here.
attention = MultiHeadAttention(config)
feedforward = FeedForward(config)

attention_output , updated_cache = attention(x)

feedforward_output = feedforward(attention_output)


p(f"attention_output : {attention_output}", f"feedforward_output : {feedforward_output}")
p(x.shape,attention_output.shape,feedforward_output.shape)

## Residual Connection

Residual Connection is a technique that adds the original input to the output of a neural network layer.

### The Formula

\[
y = F(x) + x
\]

- `x`: The original input.
- `F(x)`: The output of the layer after processing the input.
- `y`: The final output after adding the original input.

### Simple Numerical Example

Suppose we have an input:

\[
x = 10
\]

The layer processes it and produces:

\[
F(x) = 3
\]

**Without Residual Connection:**

\[
y = F(x) = 3
\]

**With Residual Connection:**

\[
y = F(x) + x = 3 + 10 = 13
\]

Instead of replacing the original information with the layer's output, we add the learned transformation to the original input.

### Why Do We Use Residual Connections?

1. **Preserve information:** The original input can pass directly to the output, reducing the risk of losing useful information during processing.
2. **Improve gradient flow:** Residual connections provide a direct path for gradients during backpropagation, helping deep neural networks train more effectively.

### Residual Connections in Transformers

In a Transformer block, residual connections are commonly used around the Attention and Feed-Forward layers.

Conceptually:

```python
x = x + attention_output
x = x + feedforward_output
```

These are simplified examples. In a standard Transformer, each sublayer's output is added to its corresponding input, and Layer Normalization is also applied according to the architecture.

**Important:** The input and output tensors must have compatible shapes for the addition to work.

### Key Takeaway

Think of a residual connection as saying:

*"Keep the original information and add the changes learned by the layer, instead of relying only on the transformed output."*

## 3. Layer normalization

LayerNorm rescales each token's feature vector independently — over the
`d_model` axis only (not batch or time):

    y = gamma * (x - mean(x)) / sqrt(var(x) + eps) + beta

`gamma`, `beta` are learned; `eps` guards division by zero.

- **Why:** keeps activations in a stable range — without it, deep stacks drift
  (values explode/vanish) during training.
- **Alternative — BatchNorm:** normalizes over the batch instead; a poor fit
  for NLP (batch statistics mix unrelated sentences, break on padded
  positions, and break with batch=1 at generation time).
- **Newer alternative — RMSNorm:** drops mean-centering, cheaper; standard in
  Llama-class models.


In [ ]:
# Deliberately wild inputs (43, 32, 5, ...): after LayerNorm every token row
# has mean ~= 0 and std ~= 1. *Relative* differences between features survive
# the rescaling and are re-scaled by the learned gamma/beta.
layer_normalization = nn.LayerNorm(config.d_model)

wild_inputs = torch.tensor([[[43,32,5,64,23,12]]],    dtype=torch.float32)

normalized_wild = layer_normalization(wild_inputs)

p(f"wild_inputs shape : {wild_inputs.shape}","\n",f"normalized_wild shape : {normalized_wild.shape}","\n",)
p(f"wild_inputs : {wild_inputs}","\n",f"normalized_wild : {normalized_wild}","\n",)

## 4. Pre-LN vs Post-LN — the GPT-2 choice

Two placements of LayerNorm around a sublayer:

    Post-LN (original paper): y = LN(x + Sublayer(x))
    Pre-LN   (GPT-2, Llama):  y = x + Sublayer(LN(x))  + one final LN at the top

- **Why pre-LN:** the residual path stays untouched (no LN inside the
  "highway"), so gradients reach layer 1 without LR-warmup tricks and deep
  stacks train stably. `TransformerBlock` is pre-LN — `layer_norm1(x)` sits
  *inside* the attention call.
- **Alternative:** post-LN gets slightly better per-layer conditioning but
  needs a careful warmup schedule (the original paper trained it that way).
- **Limitation of pre-LN:** the residual keeps accumulating, so a final top
  LayerNorm is required, otherwise output magnitudes grow with depth.
